# DCC M3 | PR #155 새 KcBERT 실험 — seed 42 실제 성능검증 (JSON 폴더 지원)

**실험 코드는 PR #155와 동일하며 변경된 부분은 데이터 입력 경로뿐입니다.** 기존 KCBERT-012와 학습/분할/임계값이 같고, 오직 관계 학습 보조 손실을 추가하는 가설입니다.

**주의:** 공식 Training 데이터를 반드시 허용된 개인 Colab 환경에서만 읽으세요. 데이터·체크포인트를 GitHub에 업로드하지 않습니다. 이 노트북은 저장소 `main` 및 제출 모델을 변경하지 않습니다.

**필수:** Colab > 런타임 > 런타임 유형 변경 > T4 GPU. 주최측의 **실제 JSON 파일이 들어 있는 공유 `TL_서울_구급` 폴더**를 Google Drive '내 드라이브'에 바로가기로 추가한 뒤 '모두 실행'합니다. (빈 이름만 같은 폴더는 안 됩니다.)

**순서:** ① 드라이브/GPU → ② 패키지 → ③ 코드 준비 → ④ 데이터 확인 → ⑤ 기본 테스트 → ⑥ seed42 학습 → ⑦ 점수 확인. 학습은 완료될 때까지 시간이 걸릴 수 있습니다.


In [ ]:
# ① Google Drive 연결 및 GPU 확인
import os
import sys
from pathlib import Path
import torch
from google.colab import drive

drive.mount("/content/drive")
if not torch.cuda.is_available():
    raise RuntimeError("GPU를 찾지 못했습니다. 런타임 → 런타임 유형 변경 → T4 GPU를 선택하세요.")
print("사용할 GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ② 필요한 도구 설치 (Colab에 설치된 PyTorch는 그대로 유지)
import subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "transformers>=4.41,<5", "huggingface-hub>=0.24,<1",
    "iterative-stratification>=0.1.9",
    "scikit-learn>=1.3,<2"
])
print("필요한 패키지를 설치했습니다.")


In [ ]:
# ③ PR #155 실험 코드(데이터 폴더 읽기 지원) 준비: 비공개 GitHub 토큰 불필요
import os,sys,hashlib
from pathlib import Path
PROJECT_ROOT = Path("/content/m3_kcbert_corr_001")
SCRIPT_PATH = PROJECT_ROOT / "scripts/lab/m3-kcbert-corr-001.py"
ENV_PATH = PROJECT_ROOT / "src/dcc_data/colab_env.py"
SCRIPT_PATH.parent.mkdir(parents=True,exist_ok=True)
ENV_PATH.parent.mkdir(parents=True,exist_ok=True)
(ENV_PATH.parent/"__init__.py").write_text("",encoding="utf-8")
SCRIPT_SOURCE = "\"\"\"M3-KCBERT-CORR-001: label-correlation training pilot based on KCBERT-012.\n\nM3-KCBERT-012 split, seeds, epochs, tokenizer, LR and output head are unchanged.\nLoss = weighted BCE + soft F1 + 0.2*pair BCE + 0.2*masked-label BCE.\\nThis is PLCP/CLCP-inspired, not an exact reproduction of the ACL paper.\nThree final-epoch seed scores use probability >= .5 without postprocessing.\nOfficial Validation is never opened.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport re\nimport shutil\nimport tempfile\nimport unicodedata\nimport zipfile\n\nEXP_ID = \"M3-KCBERT-CORR-001\"\nMODEL_ID = \"beomi/kcbert-base\"\n# Immutable safetensors addition commit, rather than the moving main ref:\n# https://huggingface.co/beomi/kcbert-base/tree/0f2f3f8\nREVISION = \"0f2f3f8\"\nLABELS = [\"고열\", \"구토\", \"두통\", \"복통\", \"어지러움\", \"열상\", \"오심\", \"전신쇠약\", \"호흡곤란\"]\nSEEDS = [42, 43, 44]\nCONFIG = dict(exp_id=EXP_ID, model=MODEL_ID, revision=REVISION, max_length=256,\n              epochs=3, lr=2e-5, dropout=0.1, batch_size=16, threshold=0.5,\n              split_seed=42, holdout_ratio=0.2, labels=LABELS,\n              weight_formula=\"clip(0.5 * sqrt(neg / max(pos, 1)), 1, 6)\",\n              baseline_source=\"M3-KCBERT-012; auxiliary pair/conditional label loss only\",\n              soft_f1_weight=1.0, soft_f1_epsilon=1e-8, threshold_operator=\">=\",\n              plcp_weight=0.2, clcp_weight=0.2, aux_training_only=True)\n\n\ndef atomic_write(path, writer):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fd, name = tempfile.mkstemp(prefix=path.name + \".\", suffix=\".tmp\", dir=path.parent)\n    os.close(fd)\n    try:\n        writer(Path(name))\n        with open(name, \"rb\") as handle:\n            os.fsync(handle.fileno())\n        os.replace(name, path)\n    finally:\n        Path(name).unlink(missing_ok=True)\n\n\ndef write_json(path, obj):\n    atomic_write(path, lambda p: p.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding=\"utf-8\"))\n\n\ndef weights(labels):\n    if not labels:\n        raise ValueError(\"Empty training split\")\n    positive = [sum(row[j] for row in labels) for j in range(len(LABELS))]\n    return [min(6.0, max(1.0, math.sqrt((len(labels) - p) / max(p, 1)) * .5)) for p in positive]\n\n\ndef transcript(doc):\n    return \" \".join(text for u in doc.get(\"utterances\", []) if isinstance(u, dict)\n                    for text in [re.sub(r\"\\s+\", \" \", u.get(\"text\", \"\")).strip()\n                                 if isinstance(u.get(\"text\", \"\"), str) else \"\"] if text)\n\n\ndef prepare_model():\n    from huggingface_hub import snapshot_download\n    from transformers import AutoModel, AutoTokenizer\n    root = Path(os.environ[\"DCC_RUN_ROOT\"]) / \"models\" / \"kcbert-base\"\n    marker = root / \"dcc_revision.json\"\n    if marker.exists():\n        if json.loads(marker.read_text()) != {\"repo_id\": MODEL_ID, \"revision\": REVISION}:\n            raise ValueError(f\"Model cache revision differs: {root}\")\n    else:\n        snapshot_download(repo_id=MODEL_ID, revision=REVISION, local_dir=str(root),\n                          allow_patterns=[\"config.json\", \"tokenizer*\", \"vocab.txt\", \"model.safetensors\"])\n    tokenizer = AutoTokenizer.from_pretrained(root, local_files_only=True)\n    # Fail before touching the data if the cached model cannot actually be loaded.\n    probe = AutoModel.from_pretrained(root, local_files_only=True)\n    del probe\n    write_json(marker, {\"repo_id\": MODEL_ID, \"revision\": REVISION})\n    return root, tokenizer\n\n\ndef prepare_rows(out, work):\n    cache = out / \"rows.json\"\n    if cache.exists():\n        return json.loads(cache.read_text(encoding=\"utf-8\"))\n    # An official folder of JSON files is supported when the shared data is not zipped.\n    # Reading is strictly read-only and preserves the baseline's sorted filename order.\n    raw_dir = os.environ.get(\"DCC_JSON_DIR\")\n    if raw_dir:\n        root = Path(raw_dir).expanduser()\n        if root.name != \"TL_서울_구급\" or not root.is_dir():\n            raise FileNotFoundError(f\"DCC_JSON_DIR must point to the TL_서울_구급 label folder: {root}\")\n        json_paths = sorted(root.rglob(\"*.json\"))\n    else:\n        from dcc_data.colab_env import find_data_root\n        source = find_data_root() / \"Training\" / \"TL_서울_구급.zip\"\n        local = work / source.name\n        atomic_write(local, lambda p: shutil.copyfile(source, p))\n        extracted = work / \"labels\"\n        extracted.mkdir(parents=True, exist_ok=True)\n        with zipfile.ZipFile(local) as archive:\n            for item in archive.infolist():\n                target = (extracted / item.filename).resolve()\n                if not target.is_relative_to(extracted.resolve()):\n                    raise ValueError(f\"Unsafe zip member: {item.filename}\")\n            archive.extractall(extracted)\n        json_paths = sorted((extracted / \"TL_서울_구급\").rglob(\"*.json\"))\n    if not json_paths:\n        raise FileNotFoundError(\"No 119 Training label JSON files were found in the selected source.\")\n    rows = []\n    for path in json_paths:\n        doc = json.loads(path.read_text(encoding=\"utf-8-sig\"))\n        symptoms = doc.get(\"symptom\", [])\n        if isinstance(symptoms, str):\n            symptoms = [symptoms]\n        symptoms = {unicodedata.normalize(\"NFC\", s.strip()) for s in symptoms if isinstance(s, str)}\n        # Repository convention: _id, otherwise the paired WAV/JSON stem.\n        call = str(doc.get(\"call_id\") or doc.get(\"_id\") or path.stem)\n        rows.append(dict(file=unicodedata.normalize(\"NFC\", path.name), call_id=call,\n                         text=transcript(doc), labels=[int(s in symptoms) for s in LABELS]))\n    if not rows:\n        raise ValueError(\"No Training label JSONs in TL_서울_구급.zip\")\n    if len({r[\"call_id\"] for r in rows}) != len(rows):\n        raise ValueError(\"Multiple rows per call: cannot preserve P0 row split without call leakage\")\n    write_json(cache, rows)\n    return rows\n\n\ndef prepare_split(out, rows):\n    import numpy as np\n    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit\n    digest = hashlib.sha256(json.dumps(rows, ensure_ascii=False, sort_keys=True).encode()).hexdigest()\n    path = out / \"split.json\"\n    if path.exists():\n        split = json.loads(path.read_text())\n        if split[\"rows_sha256\"] != digest:\n            raise ValueError(\"Cached split does not match rows\")\n    else:\n        splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=.2, random_state=42)\n        train, holdout = next(splitter.split(np.arange(len(rows)), np.array([r[\"labels\"] for r in rows])))\n        split = dict(train=train.tolist(), holdout=holdout.tolist(), rows_sha256=digest)\n        write_json(path, split)\n    train, holdout = split[\"train\"], split[\"holdout\"]\n    if not train or not holdout or sorted(train + holdout) != list(range(len(rows))):\n        raise ValueError(\"Invalid split partition\")\n    if {rows[i][\"call_id\"] for i in train} & {rows[i][\"call_id\"] for i in holdout}:\n        raise ValueError(\"call_id leakage\")\n    atomic_write(out / \"val_files.txt\", lambda p: p.write_text(\n        \"\\n\".join(sorted(rows[i][\"file\"] for i in holdout)) + \"\\n\", encoding=\"utf-8\"))\n    return train, holdout\n\n\ndef seed_all(seed, torch, np):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n\ndef loss_terms(logits, targets, criterion):\n    \"\"\"Keep both terms in fp32, including under the encoder's AMP context.\"\"\"\n    import torch\n    with torch.autocast(device_type=logits.device.type, enabled=False):\n        logits, targets = logits.float(), targets.float()\n        bce = criterion(logits, targets)\n        probabilities = logits.sigmoid()\n        soft_f1 = (2 * (probabilities * targets).sum(dim=0) /\n                   (probabilities.sum(dim=0) + targets.sum(dim=0) + 1e-8))\n        soft_f1_loss = 1 - soft_f1.mean()\n        return bce + soft_f1_loss, bce, soft_f1_loss\n\n\n\ndef observed_label_mask(targets):\n    \"\"\"Expose a random nonempty proper subset of labels per training example.\"\"\"\n    import torch\n    assert targets.ndim == 2 and targets.shape[1] >= 2\n    observed = torch.rand(targets.shape, device=targets.device) < 0.5\n    for row in range(targets.shape[0]):\n        seen = int(observed[row].sum().item())\n        if seen == 0:\n            idx = int(torch.randint(targets.shape[1], (1,), device=targets.device).item())\n            observed[row, idx] = True\n        elif seen == targets.shape[1]:\n            idx = int(torch.randint(targets.shape[1], (1,), device=targets.device).item())\n            observed[row, idx] = False\n    return observed\n\n\ndef observed_context(targets, observed):\n    \"\"\"Two channels per label: observed-positive and observed-negative.\"\"\"\n    import torch\n    shown = observed.to(targets.dtype)\n    return torch.cat((shown * targets, shown * (1.0 - targets)), dim=1)\n\n\ndef correlation_losses(representation, targets, model):\n    \"\"\"PLCP/CLCP-inspired auxiliary tasks. Never called during inference.\n\n    Revealed gold labels are fed ONLY to this auxiliary head during training.\n    The main nine-logit classifier never receives gold labels at inference.\n    \"\"\"\n    import torch\n    import torch.nn.functional as F\n    with torch.autocast(device_type=representation.device.type, enabled=False):\n        h, y = representation.float(), targets.float()\n        i, j = torch.triu_indices(y.shape[1], y.shape[1], offset=1, device=y.device)\n        pair_truth = y[:, i] * y[:, j]\n        pair_loss = F.binary_cross_entropy_with_logits(model.plcp_head(h), pair_truth)\n        observed = observed_label_mask(y)\n        context = observed_context(y, observed)\n        conditional_logits = model.clcp_head(torch.tanh(h + model.clcp_context(context)))\n        withheld = (~observed).to(y.dtype)\n        pointwise = F.binary_cross_entropy_with_logits(conditional_logits, y, reduction=\"none\")\n        conditional_loss = (pointwise * withheld).sum() / withheld.sum().clamp_min(1.0)\n        return pair_loss, conditional_loss\n\ndef run_seed(seed, out, model_path, tokens, targets, train, holdout, pos_weight):\n    import numpy as np\n    import torch\n    from transformers import AutoModel\n    from sklearn.metrics import f1_score\n    result_path = out / f\"seed_{seed}\" / \"metrics.json\"\n    if result_path.exists():\n        result = json.loads(result_path.read_text())\n        if result[\"seed\"] != seed or result[\"epoch\"] != 3 or not result_path.with_name(\"best.ckpt\").exists():\n            raise ValueError(f\"Incomplete seed artifacts: {result_path}\")\n        return result\n    seed_all(seed, torch, np)\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    amp = device.type == \"cuda\"\n\n    class Classifier(torch.nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.bert = AutoModel.from_pretrained(model_path, local_files_only=True)\n            self.dropout = torch.nn.Dropout(.1)\n            hidden = self.bert.config.hidden_size\n            self.classifier = torch.nn.Linear(hidden, len(LABELS))\n            # Training-only relationship heads; main nine-label output unchanged.\n            self.plcp_head = torch.nn.Linear(hidden, len(LABELS) * (len(LABELS) - 1) // 2)\n            self.clcp_context = torch.nn.Linear(2 * len(LABELS), hidden, bias=False)\n            self.clcp_head = torch.nn.Linear(hidden, len(LABELS))\n\n        def forward(self, ids, mask, return_representation=False):\n            h = self.dropout(self.bert(input_ids=ids, attention_mask=mask).last_hidden_state[:, 0])\n            logits = self.classifier(h)\n            return (logits, h) if return_representation else logits\n\n    model = Classifier().to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)\n    # P1 uses constant LR: no scheduler is introduced for this loss-only experiment.\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=amp)\n    generator = torch.Generator().manual_seed(seed)\n    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=device))\n    last = out / \"ckpt_last.pt\"\n    start, history = 0, []\n    if last.exists():\n        state = torch.load(last, map_location=\"cpu\", weights_only=False)\n        if state[\"config\"] != CONFIG:\n            raise ValueError(\"Checkpoint configuration mismatch\")\n        if state[\"seed\"] == seed:\n            model.load_state_dict(state[\"model\"])\n            optimizer.load_state_dict(state[\"optimizer\"])\n            scaler.load_state_dict(state[\"scaler\"])\n            start, history = state[\"epoch\"], state[\"history\"]\n            random.setstate(state[\"rng\"][\"random\"])\n            np.random.set_state(state[\"rng\"][\"numpy\"])\n            torch.set_rng_state(state[\"rng\"][\"torch\"])\n            if amp and state[\"rng\"][\"cuda\"]:\n                torch.cuda.set_rng_state_all(state[\"rng\"][\"cuda\"])\n            generator.set_state(state[\"rng\"][\"generator\"])\n    ids, masks = tokens[\"input_ids\"], tokens[\"attention_mask\"]\n    train = torch.tensor(train)\n    for epoch in range(start + 1, 4):\n        model.train()\n        order = train[torch.randperm(len(train), generator=generator)]\n        total = total_bce = total_soft_f1_loss = total_pair_loss = total_conditional_loss = 0.0\n        for step, offset in enumerate(range(0, len(order), 16)):\n            batch = order[offset:offset + 16]\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, enabled=amp):\n                logits, representation = model(\n                    ids[batch].to(device), masks[batch].to(device), return_representation=True)\n            batch_targets = targets[batch].to(device)\n            main_loss, bce, soft_f1_loss = loss_terms(logits, batch_targets, criterion)\n            pair_loss, conditional_loss = correlation_losses(representation, batch_targets, model)\n            loss = (main_loss + CONFIG[\"plcp_weight\"] * pair_loss\n                    + CONFIG[\"clcp_weight\"] * conditional_loss)\n            if not torch.isfinite(loss):\n                raise RuntimeError(f\"Non-finite loss seed={seed} epoch={epoch} step={step}\")\n            scaler.scale(loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n            total += loss.item()\n            total_bce += bce.item()\n            total_soft_f1_loss += soft_f1_loss.item()\n            total_pair_loss += pair_loss.item()\n            total_conditional_loss += conditional_loss.item()\n            if step % 200 == 0:\n                print(f\"seed={seed} epoch={epoch} step={step} loss={loss.item():.4f}\", flush=True)\n        model.eval()\n        probabilities = []\n        with torch.no_grad():\n            for offset in range(0, len(holdout), 64):\n                batch = holdout[offset:offset + 64]\n                with torch.autocast(device_type=device.type, enabled=amp):\n                    logits = model(ids[batch].to(device), masks[batch].to(device))\n                probabilities.append(logits.float().sigmoid().cpu())\n        probs = torch.cat(probabilities).numpy()\n        truth = targets[holdout].numpy()\n        pred = (probs >= .5).astype(int)\n        result = dict(seed=seed, epoch=epoch, macro_f1=float(f1_score(truth, pred, average=\"macro\", zero_division=0)),\n                      micro_f1=float(f1_score(truth, pred, average=\"micro\", zero_division=0)),\n                      per_symptom_f1=dict(zip(LABELS, f1_score(truth, pred, average=None, zero_division=0).tolist())),\n                      empty_predictions=int((pred.sum(axis=1) == 0).sum()),\n                      train_loss=total / (step + 1),\n                      train_bce=total_bce / (step + 1),\n                      train_soft_f1_loss=total_soft_f1_loss / (step + 1),\n                      train_soft_f1=1 - total_soft_f1_loss / (step + 1),\n                      train_plcp_loss=total_pair_loss / (step + 1),\n                      train_clcp_loss=total_conditional_loss / (step + 1))\n        history.append(result)\n        state = dict(config=CONFIG, seed=seed, epoch=epoch, model=model.state_dict(),\n                     optimizer=optimizer.state_dict(), scheduler=None, scaler=scaler.state_dict(), history=history,\n                     rng=dict(random=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),\n                              cuda=torch.cuda.get_rng_state_all() if amp else [], generator=generator.get_state()))\n        atomic_write(last, lambda p: torch.save(state, p))\n        write_json(result_path.with_name(\"history.json\"), history)\n        print(json.dumps(result, ensure_ascii=False), flush=True)\n    # Final epoch is the experiment score; no epoch or threshold selection.\n    final = dict(config=CONFIG, seed=seed, epoch=3, model=model.state_dict(),\n                 model_local_path=str(model_path), pos_weight=pos_weight)\n    atomic_write(result_path.with_name(\"best.ckpt\"), lambda p: torch.save(final, p))\n    write_json(result_path, history[-1])\n    return history[-1]\n\n\ndef smoke():\n    # Synthetic loss checks only; no model download or training.\n    try:\n        import torch\n    except ModuleNotFoundError as exc:\n        if exc.name != \"torch\":\n            raise\n        print(\"smoke: torch unavailable; tensor loss/gradient checks skipped\", flush=True)\n    else:\n        logits = torch.tensor([[0., 2., -2.]] * 2, requires_grad=True)\n        targets = torch.tensor([[1., 0., 1.], [0., 1., 0.]])\n        criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([1., 2., 3.]))\n        loss, bce, soft_loss = loss_terms(logits, targets, criterion)\n        probabilities = logits.detach().sigmoid()\n        expected = 1 - (2 * (probabilities * targets).sum(0) /\n                        (probabilities.sum(0) + targets.sum(0) + 1e-8)).mean()\n        assert loss.dtype == torch.float32\n        assert torch.allclose(soft_loss, expected)\n        assert torch.allclose(loss, criterion(logits, targets) + expected)\n        loss.backward()\n        assert torch.isfinite(logits.grad).all()\n        empty = torch.zeros_like(targets)\n        assert torch.isfinite(loss_terms(logits.detach(), empty, criterion)[0])\n        assert (torch.tensor([.49, .5, .51]) >= .5).tolist() == [False, True, True]\n        # Synthetic checks: hidden gold labels must never enter conditional context.\n        torch.manual_seed(42)\n        y = torch.tensor([[1., 0., 1., 0., 0., 1., 0., 1., 0.],\n                          [0., 0., 0., 1., 0., 0., 0., 0., 1.]])\n        observed = observed_label_mask(y)\n        assert observed.any(dim=1).all() and (~observed).any(dim=1).all()\n        changed = y.clone()\n        changed[~observed] = 1 - changed[~observed]\n        assert torch.equal(observed_context(y, observed), observed_context(changed, observed))\n        class Heads(torch.nn.Module):\n            def __init__(self):\n                super().__init__()\n                self.plcp_head = torch.nn.Linear(5, 36)\n                self.clcp_context = torch.nn.Linear(18, 5, bias=False)\n                self.clcp_head = torch.nn.Linear(5, 9)\n        heads = Heads()\n        embedding = torch.randn(2, 5, requires_grad=True)\n        plcp, clcp = correlation_losses(embedding, y, heads)\n        assert plcp.isfinite() and clcp.isfinite()\n        (plcp + clcp).backward()\n        assert embedding.grad is not None and embedding.grad.abs().sum() > 0\n        assert all(p.grad is not None and p.grad.isfinite().all()\n                   for p in heads.parameters())\n    rows = [[1] * 9] + [[0] * 9] * 99\n    assert weights(rows) == [math.sqrt(99) * .5] * 9\n    assert weights([[0] * 9] * 1000) == [6.] * 9\n    assert weights([[1] * 9] * 2) == [1.] * 9\n    assert transcript({\"utterances\": [{\"text\": \" 가\\n 나 \"}, {\"text\": \"다\", \"Speaker\": \"ignored\"}]}) == \"가 나 다\"\n    with tempfile.TemporaryDirectory() as name:\n        p = Path(name) / \"check.json\"\n        write_json(p, {\"exp_id\": EXP_ID})\n        assert json.loads(p.read_text())[\"exp_id\"] == EXP_ID\n        assert list(p.parent.iterdir()) == [p]\n    print(\"smoke OK: base and correlation loss, masked-label no-leakage, gradients, text only; no training\")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--out\", default=os.environ.get(\"LAB_OUT\"))\n    parser.add_argument(\"--smoke\", action=\"store_true\")\n    parser.add_argument(\"--seeds\", nargs=\"+\", type=int, default=SEEDS,\n                        help=\"Run a single seed (42) for a pilot before a full 3-seed run.\")\n    args = parser.parse_args()\n    if args.smoke:\n        smoke()\n        return\n    if not args.out:\n        parser.error(\"--out or LAB_OUT required\")\n    if not args.seeds or len(set(args.seeds)) != len(args.seeds) or any(seed not in SEEDS for seed in args.seeds):\n        parser.error(\"--seeds must be unique values from 42, 43, 44.\")\n    if os.environ.get(\"LAB_EXP_ID\", EXP_ID) != EXP_ID:\n        raise ValueError(\"LAB_EXP_ID mismatch\")\n    model_path, tokenizer = prepare_model()  # Before all data preparation.\n    import torch\n    import numpy as np\n    out = Path(args.out)\n    work = Path(os.environ[\"LAB_WORK\"]) / EXP_ID.lower()\n    out.mkdir(parents=True, exist_ok=True)\n    work.mkdir(parents=True, exist_ok=True)\n    run_config = {**CONFIG, \"training_seeds\": args.seeds}\n    if (out / \"config.json\").exists() and json.loads((out / \"config.json\").read_text()) != run_config:\n        raise ValueError(\"Output directory belongs to a different configuration\")\n    write_json(out / \"config.json\", run_config)\n    if (out / \"metrics.json\").exists():\n        if not (out / \"best.ckpt\").exists():\n            raise ValueError(\"Completed metrics exist without best.ckpt\")\n        print(\"All seeds complete; reusing metrics.json\", flush=True)\n        return\n    rows = prepare_rows(out, work)\n    train, holdout = prepare_split(out, rows)\n    pos_weight = weights([rows[i][\"labels\"] for i in train])\n    write_json(out / \"pos_weight.json\", dict(formula=CONFIG[\"weight_formula\"], weights=dict(zip(LABELS, pos_weight)),\n                                              train_positive_counts=np.array([rows[i][\"labels\"] for i in train]).sum(0).tolist()))\n    cache = out / \"tokens.pt\"\n    if cache.exists():\n        tokens = torch.load(cache, map_location=\"cpu\", weights_only=True)\n    else:\n        tokens = dict(tokenizer([r[\"text\"] for r in rows], padding=\"max_length\", truncation=True,\n                                max_length=256, return_tensors=\"pt\"))\n        atomic_write(cache, lambda p: torch.save(tokens, p))\n    targets = torch.tensor([r[\"labels\"] for r in rows], dtype=torch.float32)\n    results = [run_seed(seed, out, model_path, tokens, targets, train, holdout, pos_weight) for seed in args.seeds]\n    # Deterministic representative (first selected seed), not chosen by holdout score.\n    representative_seed = args.seeds[0]\n    atomic_write(out / \"best.ckpt\", lambda p: shutil.copyfile(out / f\"seed_{representative_seed}\" / \"best.ckpt\", p))\n    mean_macro = sum(r[\"macro_f1\"] for r in results) / len(results)\n    metrics = dict(exp_id=EXP_ID, primary_metric=dict(name=\"macro_f1\", value=mean_macro, split=\"internal_holdout\"),\n                   metrics=dict(seeds=results, mean_micro_f1=sum(r[\"micro_f1\"] for r in results) / len(results),\n                                mean_per_symptom_f1={s: sum(r[\"per_symptom_f1\"][s] for r in results) / len(results) for s in LABELS}),\n                   seed=representative_seed, seeds=args.seeds, train_rows=len(train), eval_rows=len(holdout), official_val=None,\n                   ckpt=\"best.ckpt\", notes=\"Final epoch 3; threshold >=0.5 fixed; empty predictions unchanged; first-seed representative checkpoint. \"\n                   \"M3-KCBERT-012 split reused; BCE + soft macro-F1 + training-only 0.2 PLCP and 0.2 CLCP; \"\n                   \"weights computed from Training partition only; official Validation unused.\")\n    write_json(out / \"metrics.json\", metrics)\n\n\nif __name__ == \"__main__\":\n    main()\n"
DATA_ENV_SOURCE = "\"\"\"Locate the official data root and the run-output root the same way on every machine.\n\nTeammates only need the Drive shortcuts described in ``docs/team_colab_setup.md``;\nno path has to be edited in a notebook or script.\n\nData root precedence: explicit argument -> ``DCC_DATA_ROOT`` (``M1_DATA_ROOT`` legacy)\n-> repository ``data/raw_original`` -> Colab Drive auto-discovery.\nRun root precedence: explicit argument -> ``DCC_RUN_ROOT`` -> Colab ``MyDrive/dcc_runs``\n-> repository ``outputs``.\n\"\"\"\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nDATA_DIR_NAME = \"대학부 데이터\"\nRUN_DIR_NAME = \"dcc_runs\"\nDRIVE_MOUNT = Path(\"/content/drive\")\n_REPO_ROOT = Path(__file__).resolve().parents[2]\n\n\ndef is_data_root(path: Path) -> bool:\n    \"\"\"A data root holds the official ``Training`` and ``Validation`` folders directly.\"\"\"\n    return (path / \"Training\").is_dir() and (path / \"Validation\").is_dir()\n\n\ndef _unwrap(path: Path) -> Path | None:\n    # Drive downloads often nest the folder once more (대학부 데이터/대학부 데이터).\n    for candidate in (path, path / DATA_DIR_NAME):\n        if candidate.is_dir() and is_data_root(candidate):\n            return candidate\n    return None\n\n\ndef mount_drive_if_colab() -> bool:\n    \"\"\"Mount Drive on Colab (no-op elsewhere). Returns True if MyDrive is available.\"\"\"\n    if (DRIVE_MOUNT / \"MyDrive\").is_dir():\n        return True\n    try:\n        from google.colab import drive  # type: ignore\n    except ImportError:\n        return False\n    drive.mount(str(DRIVE_MOUNT))\n    return (DRIVE_MOUNT / \"MyDrive\").is_dir()\n\n\ndef _drive_candidates() -> list[Path]:\n    # Shallow lookups only: walking a whole Drive mount is very slow.\n    base = DRIVE_MOUNT\n    found = [base / \"MyDrive\" / DATA_DIR_NAME]\n    found += sorted(base.glob(f\".shortcut-targets-by-id/*/{DATA_DIR_NAME}\"))\n    found += sorted(base.glob(f\"Shareddrives/*/{DATA_DIR_NAME}\"))\n    found += sorted(base.glob(f\"Shared drives/*/{DATA_DIR_NAME}\"))\n    return found\n\n\ndef find_data_root(explicit: str | os.PathLike | None = None) -> Path:\n    \"\"\"Return the folder that contains ``Training/`` and ``Validation/``.\"\"\"\n    tried: list[str] = []\n\n    def check(label: str, raw) -> Path | None:\n        path = Path(raw).expanduser()\n        tried.append(f\"{label}: {path}\")\n        return _unwrap(path)\n\n    if explicit:\n        # An explicit choice must be honoured or fail loudly, never silently replaced.\n        hit = check(\"argument\", explicit)\n        if hit:\n            return hit\n        raise FileNotFoundError(f\"Training/Validation not found under explicit data root.\\n  \" + \"\\n  \".join(tried))\n    for name in (\"DCC_DATA_ROOT\", \"M1_DATA_ROOT\"):\n        if os.environ.get(name):\n            hit = check(f\"${name}\", os.environ[name])\n            if hit:\n                return hit\n            raise FileNotFoundError(f\"${name} is set but has no Training/Validation.\\n  \" + \"\\n  \".join(tried))\n    hit = check(\"repo default\", _REPO_ROOT / \"data\" / \"raw_original\")\n    if hit:\n        return hit\n    if mount_drive_if_colab():\n        for candidate in _drive_candidates():\n            hit = check(\"drive\", candidate)\n            if hit:\n                return hit\n    raise FileNotFoundError(\n        f\"Official data folder '{DATA_DIR_NAME}' (with Training/ and Validation/) was not found.\\n  \"\n        + \"\\n  \".join(tried)\n        + f\"\\nColab: add a Drive shortcut named '{DATA_DIR_NAME}' under My Drive \"\n        \"(see docs/team_colab_setup.md). Local: set DCC_DATA_ROOT.\"\n    )\n\n\ndef find_run_root(explicit: str | os.PathLike | None = None, create: bool = True) -> Path:\n    \"\"\"Return the folder that holds checkpoints and logs (Drive on Colab, ``outputs`` locally).\"\"\"\n    if explicit:\n        root = Path(explicit).expanduser()\n    elif os.environ.get(\"DCC_RUN_ROOT\"):\n        root = Path(os.environ[\"DCC_RUN_ROOT\"]).expanduser()\n    elif mount_drive_if_colab():\n        root = DRIVE_MOUNT / \"MyDrive\" / RUN_DIR_NAME\n    else:\n        root = _REPO_ROOT / \"outputs\"\n    if create:\n        root.mkdir(parents=True, exist_ok=True)\n    return root\n"
SCRIPT_PATH.write_text(SCRIPT_SOURCE,encoding="utf-8")
ENV_PATH.write_text(DATA_ENV_SOURCE,encoding="utf-8")
sys.path.insert(0,str(PROJECT_ROOT/"src"))
os.environ["PYTHONPATH"]=str(PROJECT_ROOT/"src")+os.pathsep+os.environ.get("PYTHONPATH","")
print("실험 코드 준비 완료",SCRIPT_PATH,"SHA256:",hashlib.sha256(SCRIPT_PATH.read_bytes()).hexdigest())


In [ ]:
# ④ 공유된 공식 Training JSON 폴더 탐색. 원본 데이터는 수정·복사·공개하지 않습니다.
import os
from pathlib import Path
MOUNT = Path("/content/drive")
MYDRIVE = MOUNT/"MyDrive"
# 자동 탐색에 실패한 경우에만 파일 창에서 폴더 경로를 복사하여 입력하세요.
MANUAL_JSON_DIR = ""

roots = [MYDRIVE, MYDRIVE/"대학부 데이터", MYDRIVE/"2026 DCC 대학부 데이터",
         MYDRIVE/"2026 DCC 대학부 데이터"/"대학부 데이터"]
for base in [MOUNT/".shortcut-targets-by-id",MOUNT/"Shareddrives",MOUNT/"Shared drives"]:
    if base.is_dir():
        roots.extend(sorted(base.iterdir()))
suffixes = [
    "", "TL_서울_구급", "2.라벨링데이터/TL_서울_구급",
    "Training/2.라벨링데이터/TL_서울_구급",
    "Training/TL_서울_구급",
    "대학부 데이터/Training/2.라벨링데이터/TL_서울_구급",
    "대학부 데이터/Training/TL_서울_구급",
    "2026 DCC 대학부 데이터/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급",
]
candidates = [Path(MANUAL_JSON_DIR).expanduser()] if MANUAL_JSON_DIR.strip() else [
    root/suffix for root in roots for suffix in suffixes]
selected=None
for folder in candidates:
    if folder.name=="TL_서울_구급" and folder.is_dir() and next(folder.glob("*.json"),None):
        selected=folder
        break
if selected is None:
    raise FileNotFoundError(
      "JSON 파일이 들어 있는 TL_서울_구급 폴더를 찾지 못했습니다. "
      "주최측의 실제 데이터가 들어 있는 공유 TL_서울_구급 폴더를 "
      "Google Drive → 내 드라이브에 바로가기 추가한 뒤 셀 ④를 재실행하세요. "
      "또는 MANUAL_JSON_DIR에 Colab Files 창의 전체 폴더 경로를 입력하세요."
    )
file_count=sum(1 for _ in selected.glob("*.json"))
print("Training JSON 폴더:",selected)
print("확인한 JSON 파일 수:",file_count)
if file_count<100:
    raise RuntimeError("Training 파일이 100개 미만입니다. 원본 전체가 연결됐는지 확인하세요. 일부 데이터로는 기존 성능 비교 불가.")
os.environ["DCC_JSON_DIR"]=str(selected)
os.environ.pop("DCC_DATA_ROOT",None)
PERSONAL_RUN_ROOT=MYDRIVE/"dcc_runs_me"
LAB_OUT=PERSONAL_RUN_ROOT/"lab"/"M3-KCBERT-CORR-001-seed42-json"
LAB_WORK=Path("/content/m3_kcbert_corr_work")
PERSONAL_RUN_ROOT.mkdir(parents=True,exist_ok=True)
LAB_WORK.mkdir(parents=True,exist_ok=True)
os.environ.update(DCC_RUN_ROOT=str(PERSONAL_RUN_ROOT),LAB_OUT=str(LAB_OUT),
                  LAB_WORK=str(LAB_WORK),LAB_EXP_ID="M3-KCBERT-CORR-001")
print("출력 위치:",LAB_OUT)
print("검증 전용 Training 분할 80:20. 공식 Validation은 읽지 않습니다.")


In [ ]:
# ⑤ 1분 이내 작은 자동검사 (정답 라벨이 보조 입력으로 새지 않는지 등 검사)
import subprocess
subprocess.run([sys.executable, str(SCRIPT_PATH), "--smoke"], check=True, env=os.environ.copy())
print("작은 테스트 완료. 이제 실제 학습을 실행할 수 있습니다.")


In [ ]:
# ⑥ 실제 학습: seed 42 한 번, epoch 3회 (GPU·시간 사용)
# 실행 중 런타임이 끊길 수 있습니다. 결과는 개인 Drive의 LAB_OUT에 저장됩니다.
RUN_TRAINING = True
if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("GPU를 찾지 못했습니다.")
    command = [
        sys.executable, "-u", str(SCRIPT_PATH),
        "--out", str(LAB_OUT),
        "--seeds", "42",
    ]
    print("실행:", " ".join(command))
    subprocess.run(command, check=True, env=os.environ.copy())
else:
    print("학습을 실행하지 않았습니다.")


In [ ]:
# ⑦ 성능 분석. 완료 전에는 수치가 나오지 않습니다.
import json
metrics_file=LAB_OUT/"metrics.json"
if not metrics_file.is_file():
    print("아직 학습 결과가 없습니다. 셀 ⑥을 확인하세요.")
else:
    result=json.loads(metrics_file.read_text(encoding="utf-8"))
    score=result["metrics"]["seeds"][0]
    current=float(score["macro_f1"])
    reference=0.6337  # KCBERT-012 seed42, 팀 실험기록의 참고값
    print("실험:",result["exp_id"],"학습 seed:",score["seed"])
    print("새 모델 내부 holdout Macro F1:",round(current,5))
    print("기존 KCBERT-012 기록 seed42 참고값:",reference)
    print("참고값과의 차이:",round(current-reference,5))
    print("Micro F1:",round(float(score["micro_f1"]),5))
    print("증상별 F1:",json.dumps(score["per_symptom_f1"],ensure_ascii=False,indent=2))
    print("평가 파일 수:",result["eval_rows"],"Train 파일 수:",result["train_rows"])
    print("같은 전체 데이터·split일 때만 기존 기록과 직접 비교 가능합니다.")
    print("이 점수 하나만으로 모델을 채택하지 말고, 개선 가능성이 보이면 3-seed를 별도 폴더에서 실행하세요.")
    print("결과 JSON:",metrics_file)
